In [1]:
%load_ext autoreload
%autoreload 2

from utils.iv import construct_iv_surface, create_iv_surface_plot
from datetime import datetime, timezone

This took 08:42 due to massive memory usage ballooning to 50+ gb. Let's get that down.

I tried reducing max_workers to 3, which actually made things worse. Now it takes 13:17.

With chunking implemented, we're down to 06:17, which is acceptable.

In [21]:
# Construct IV surface data
iv_df = construct_iv_surface(
    inst_family='BTC-USD',
    start_date=datetime(2025, 1, 15, tzinfo=timezone.utc),
    num_days=1,
    time_step_minutes=60,
    verbose=True,
    diagnostics=True,  # Enable failure diagnostics
    use_bid_ask=True
)

Processing dates:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching FUTURES data (module=6) for BTC-USD
Period: 2025-01-15 00:00:00+00:00 to 2025-01-15 23:59:59+00:00
Split into 1 requests


Fetching data:   0%|          | 0/1 [00:00<?, ?it/s]

Fetch #1/1: 6 files found | Total: 6 files, 914.15 MB


✓ Successfully fetched 144 records
  2025-01-15: 6 available expiries
Fetching OPTION data (module=6) for BTC-USD
Period: 2025-01-15 00:00:00+00:00 to 2025-01-15 23:59:59+00:00
Split into 1 requests


Fetching data:   0%|          | 0/1 [00:00<?, ?it/s]

Fetch #1/1: 384 files found | Total: 384 files, 195.90 MB


✓ Successfully fetched 8776 records

    ═══ IV Solver Diagnostics ═══
    Total options: 8,776
    ├─ Rejected (below intrinsic): 1,256 (14.3%)
    ├─ Converged successfully: 4,274 (48.7%)
    ├─ Accepted near-convergence: 61 (0.7%)
    └─ Failed to converge: 3,185 (36.3%)

    Failure Analysis:
    ├─ Low vega (>50% iters): 3,185 (100.0%)
    ├─ Hit lower bound (0.01): 870 (27.3%)
    ├─ Hit upper bound (5.0): 312 (9.8%)
    └─ Hit max iterations: 3,185 (100.0%)

    Final relative errors (failed entries):
    ├─ Mean: 80.24%
    ├─ Median: 100.00%
    └─ 90th percentile: 100.00%

    Sample failures (first 3):
    #1: C, K=75000, F=97203, T=0.006y
        Moneyness: -0.259, Market: $22065.15, Intrinsic: $22203.30
        Final σ: 0.2233, Final price: $22203.30, Error: 0.63%
        Iterations: 100, Low vega count: 98
    #2: C, K=75000, F=97230, T=0.005y
        Moneyness: -0.260, Market: $22168.47, Intrinsic: $22230.15
        Final σ: 0.0100, Final price: $22230.15, Error: 0.28%
 

In [22]:
total_rows = len(iv_df)

# Count rows in each IV range
point3_rows = len(iv_df[(iv_df['bid_iv'] == 0.3) | (iv_df['ask_iv'] == 0.3)])
mid_range_rows = len(iv_df[((iv_df['bid_iv'] >= 0.05) & (iv_df['bid_iv'] < 0.3)) | ((iv_df['ask_iv'] >= 0.05) & (iv_df['ask_iv'] < 0.3))])
low_range_rows = len(iv_df[((iv_df['bid_iv'] >= 0.005) & (iv_df['bid_iv'] < 0.05)) | ((iv_df['ask_iv'] >= 0.005) & (iv_df['ask_iv'] < 0.05))])
very_low_rows = len(iv_df[(iv_df['bid_iv'] < 0.005) | (iv_df['ask_iv'] < 0.005)])
na_rows = len(iv_df[iv_df['bid_iv'].isna() | iv_df['ask_iv'].isna()])

# Print results with percentages
print(f"Rows with NA: {na_rows} ({na_rows/total_rows*100:.2f}%)")
print(f"Rows with IV < 0.005: {very_low_rows} ({very_low_rows/total_rows*100:.2f}%)")
print(f"Rows with 0.005 <= IV < 0.05: {low_range_rows} ({low_range_rows/total_rows*100:.2f}%)")
print(f"Rows with 0.05 <= IV < 0.3: {mid_range_rows} ({mid_range_rows/total_rows*100:.2f}%)")
print(f"Rows with IV = 0.3: {point3_rows} ({point3_rows/total_rows*100:.2f}%)")

for col in iv_df.columns:
    print(col)


Rows with NA: 1090 (20.11%)
Rows with IV < 0.005: 0 (0.00%)
Rows with 0.005 <= IV < 0.05: 51 (0.94%)
Rows with 0.05 <= IV < 0.3: 17 (0.31%)
Rows with IV = 0.3: 0 (0.00%)
timestamp
symbol
expiry
strike
option_type
bid_1_px
forward_price
tenor_days
log_moneyness
bid_iv
ask_1_px
ask_iv


In [ ]:
# Create interactive plot
fig = create_iv_surface_plot(iv_df, grid_resolution=40, kernel='quintic', smoothing=0.1)
fig.show()

Processing frames:   0%|          | 0/24 [00:00<?, ?it/s]


DEBUG _fit_rbf_surface:
  Input IVs: count=191, range=[0.0100, 0.8236]
  Log moneyness range: [-1.6308, 0.5664]
  Tenor range: [2.33, 163.33] days
  Kernel: cubic, Smoothing: 0.1
  RBF fitted successfully
  Evaluation grid: 40x40 = 1600 points
  Output IV range (clamped): [0.0783, 0.7672]
  Output IV surface: range=[0.0783, 0.7672]

✓ Generated 24 frames with 48 total surfaces
First frame has 4 traces:
  - Bid IV: Surface
    Shape: (40, 40), Range: [0.0783, 0.7672]
  - Bid Points: Scatter3d
  - Ask IV: Surface
    Shape: (40, 40), Range: [0.5323, 3.6249]
  - Ask Points: Scatter3d


In [26]:
import importlib
import utils.iv
importlib.reload(utils.iv)
from utils.iv import construct_iv_surface, create_iv_surface_plot